In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder , StandardScaler , LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline ,Pipeline 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier 
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
from pandas.plotting import scatter_matrix
sys.path.append("../src")
sys.path.append("../data/")
from utils import *

In [ ]:
data = pd.read_csv("../data/AdultCensusIncomeDataset/adult.csv")
data = data.replace('?', np.nan)
DT_MD = 3
NSTIM = 500
RS = 42
DLFXD = ['income']

In [ ]:
X = data.drop(DLFXD , axis=1) ; y = data['income'].copy().to_frame()
X_categorial_Types = X.select_dtypes(include=['object']).columns ; X_numerical_types = X.select_dtypes(include=['int64' , 'float64']).columns

In [ ]:
numerical_pipeline  = Pipeline([
    ('imputer' , SimpleImputer(strategy='median')),
    ('scaler' , StandardScaler())
])

categorial_pipeline = Pipeline([
    ('imputer' , SimpleImputer(strategy='most_frequent')),
    ('cat_to_num' , OrdinalEncoder(handle_unknown='error'))
])
Y_pipeline          = Pipeline([
    ('imputer' , SimpleImputer(strategy='most_frequent')),
    ('cat' ,  OrdinalEncoder(handle_unknown='error'))
])

In [ ]:
preprocessing_pipeline_Xv = ColumnTransformer([
    ('cat' , categorial_pipeline , X_categorial_Types),
    ('num', numerical_pipeline , X_numerical_types)
])
preprocessing_pipeline_Yv = ColumnTransformer([
    ('num', Y_pipeline , ['income'])
])

In [ ]:
y = preprocessing_pipeline_Yv.fit_transform(y);X = preprocessing_pipeline_Xv.fit_transform(X)
X_train , X_valid , y_train , y_valid = train_test_split(X , y[: , 0] , test_size=0.2 , stratify=data['income'] , random_state=RS)

In [ ]:
tree_classifier = DecisionTreeClassifier(max_depth=DT_MD , random_state=RS)
rf_classifier = RandomForestClassifier(NSTIM , max_depth=DT_MD , random_state=RS)
gb_classifier = GradientBoostingClassifier(n_estimators=NSTIM , max_depth=DT_MD , random_state=RS)
lr_classifier = LogisticRegression()

In [ ]:
vote_classifier = VotingClassifier([
    # ('rf' , rf_classifier),
    # ('tree' , tree_classifier),
    # ('lr' , lr_classifier),
    ('gb' , gb_classifier)
])

In [ ]:
vote_classifier.fit(X_train , y_train)

In [ ]:
y_pred = vote_classifier.predict(X_valid)

In [ ]:
print_classification_metrics(y_valid , y_pred)

In [ ]:
plot_roc_curve(y_valid , y_pred , save=Path(f"../tests/Project_1_plots/roc_curve_plot_pipelinev2_Gradientboosting_seed_{RS}.png"))
plot_roc_curve_with_auc(y_valid , y_pred , save=Path(f"../tests/Project_1_plots/roc_curve_with_auc_plot_pipelinev2_Gradientboosting_seed_{RS}.png"))
plot_pr_curve_with_ap(y_valid , y_pred , save=Path(f"../tests/Project_1_plots/pr_curve_with_ap_plot_pipelinev2_Gradientboosting_seed_{RS}.png"))

# Runs

#### run with linear model and standard scaler and nothing else
- -> Accuracy:  0.8285
- -> Precision: 0.7235
- -> Recall:    0.4656
- -> F1-Score:  0.5666
#### run with linear model and standard scaler and with pipeline version 2
- -> Accuracy:  0.8239
- -> Precision: 0.7086
- -> Recall:    0.4560
- -> F1-Score:  0.5549
#### run with voting model and standard scaler and with pipeline version 2
- -> Accuracy:  0.8406
- -> Precision: 0.8451
- -> Recall:    0.4139
- -> F1-Score:  0.5557
#### run with gradient boosting classifier and standard scaler and with pipeline version 2
- -> Accuracy:  0.8689
- -> Precision: 0.7738
- -> Recall:    0.6435
- -> F1-Score:  0.7026